# BPL Study Craft

Shared BPL setup for this folder and its subfolders. Code runs automatically when a dialogue opens. Run `setup_bpl()` to refresh the keyboard and toolbar. Browser behaviour is in CRAFT.js and styles in CRAFT.css.

Installation is handled below; no requirements file is needed. BPL_VERSION selects the tested release (0.1.61). Plain pip install basedpl installs the latest compatible release on a fresh setup but leaves an existing installation alone. pip install --upgrade basedpl upgrades it. This Craft deliberately uses the tested version because the keyboard patches and generated browser catalogue depend on it.

To check installed and available versions without changing anything, run check_bpl_versions(). Before adopting another release, update BPL_VERSION, check the keyboard patches, rebuild_bpl_browser(), and test in a new dialogue.


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import subprocess
import sys
from IPython import get_ipython
from IPython.display import HTML, Javascript, display
from dialoghelper.core import realpath

BPL_VERSION = '0.1.61'  # Tested with the keyboard integration below.

try:
    _bpl_version = version('basedpl')
except PackageNotFoundError:
    _bpl_version = None
if _bpl_version != BPL_VERSION:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'basedpl=={BPL_VERSION}', '-q'])
if get_ipython().find_line_magic('bpl') is None:
    get_ipython().run_line_magic('load_ext', 'basedpl.notebooks')
_bpl_folder = Path(await realpath(''))
_bpl_craft_dir = next(p for p in [_bpl_folder, *_bpl_folder.parents] if (p / 'CRAFT.js').is_file())


In [ ]:
def check_bpl_versions():
    """List installed and available BPL releases without installing anything."""
    subprocess.check_call([sys.executable, '-m', 'pip', 'index', 'versions', 'basedpl'])


In [ ]:
bpl_examples = {'⍳': [{'label': 'One input · zero-based indices', 'code': '⍳4'}, {'label': 'Two inputs · index of', 'code': '"abc"⍳"cabz"'}], '↦': [{'label': 'One input · range from one', 'code': '↦4'}, {'label': 'Two inputs · inclusive range', 'code': '0↦4'}], '=': [{'label': 'One input · classify equal items', 'code': '=4 7 4 9'}, {'label': 'Two inputs · compare each item', 'code': '1 2 3 = 1 9 3'}], '≠': [{'label': 'One input · first occurrences', 'code': '≠4 7 4 9'}, {'label': 'Two inputs · not equal', 'code': '1 2 3 ≠ 1 9 3'}], '≡': [{'label': 'One input · nesting depth', 'code': '≡1 2 3'}, {'label': 'Two inputs · whole-array match', 'code': '1 2 3 ≡ 1 2 3'}], '≢': [{'label': 'One input · count items', 'code': '≢2 4 9'}, {'label': 'Two inputs · whole-array mismatch', 'code': '1 2 3 ≢ 1 2'}], '⍴': [{'label': 'One input · shape', 'code': '⍴1 2 3'}, {'label': 'Two inputs · reshape', 'code': '2 3⍴⍳6'}], '×': [{'label': 'One input · sign', 'code': '×¯3 0 4'}, {'label': 'Two inputs · multiply', 'code': '3×1 2 3'}], '÷': [{'label': 'One input · reciprocal', 'code': '÷1 2 4'}, {'label': 'Two inputs · divide', 'code': '12÷3'}], '+': [{'label': 'Two inputs · add', 'code': '1 2 3 + 10'}], '-': [{'label': 'One input · negate', 'code': '-1 2 3'}, {'label': 'Two inputs · subtract', 'code': '10-3'}], '/': [{'label': 'Reduce · sum a list', 'code': '+/2 4 9'}, {'label': 'Reduce · product', 'code': '×/1 2 3 4'}], '\\': [{'label': 'Scan · running sum', 'code': '+\\1 2 3 4'}], '⌽': [{'label': 'One input · reverse', 'code': '⌽1 2 3'}, {'label': 'Two inputs · rotate', 'code': '1⌽1 2 3'}], '↑': [{'label': 'One input · first', 'code': '↑10 20 30'}, {'label': 'Two inputs · take', 'code': '2↑10 20 30'}], '↓': [{'label': 'Two inputs · drop', 'code': '1↓10 20 30'}], '⎕': [{'label': 'Display a result', 'code': '⎕←1 2 3'}], '⍟': [{'label': 'One input · natural logarithm', 'code': '⍟1 2 4'}, {'label': 'Two inputs · logarithm base two', 'code': '2⍟8'}], '√': [{'label': 'One input · square root', 'code': '√1 4 9'}, {'label': 'Two inputs · cube root', 'code': '3√27'}]}

In [ ]:
def _bpl_keyboard():

    # Run after loading basedpl.notebooks to enable keyboard symbol selection.
    import json
    from importlib.resources import files
    from basedpl import symbols
    from IPython import get_ipython
    from IPython.display import HTML, Javascript, display

    def _patch(source, old, new):
        if source.count(old) != 1: raise RuntimeError('This symbol setup expects BPL 0.1.61.')
        return source.replace(old, new)

    _assets = files('basedpl')
    _lb = (_assets / 'lb.js').read_text()
    _input = (_assets / 'input.js').read_text()
    _lb = _patch(_lb, 'let overlay = false, active, lastEditor, choice, keyInput = false;', '''let overlay = false, active, lastEditor, choice, keyInput = false, picked = 0;
        function highlight() {
            tip.setAttribute('role', 'listbox');
            [...tip.querySelectorAll('button')].forEach((b, i) => {
                b.setAttribute('role', 'option'); b.setAttribute('aria-selected', String(i === picked));
            });
        }''')
    _lb = _patch(_lb, 'tip.replaceChildren();', 'picked = 0; tip.replaceChildren(); tip.scrollTop = 0;')
    _lb = _patch(_lb, 'tip.hidden = false;', 'tip.hidden = false; if (!anchor) highlight();')
    _lb = _patch(_lb, 'const typed = active?.id === e.id && active.start === item?.start;', '''const typed = active?.id === e.id && active.start === item?.start;
            if (typed && plain && !ev.shiftKey && !tip.hidden && !choice?.anchor &&
                ['ArrowUp', 'ArrowDown', 'Enter', 'Tab', 'Escape'].includes(ev.key)) {
                const buttons = [...tip.querySelectorAll('button')];
                if (ev.key === 'Escape') cancel();
                else if (ev.key === 'ArrowUp' || ev.key === 'ArrowDown') {
                    if (buttons.length) picked = (picked + (ev.key === 'ArrowDown' ? 1 : -1) + buttons.length) % buttons.length;
                    highlight(); buttons[picked]?.scrollIntoView({block:'nearest'});
                } else {
                    if (buttons.length) e.insert(buttons[picked].dataset.glyph, item.start);
                    cancel();
                }
                ev.preventDefault(); ev.stopImmediatePropagation(); return;
            }''')
    _input = _patch(_input, r'^%%(?:bpl|apl)[^\S\n]*', r'^%%(?:bpl|apl)(?:[ \t]+--animate)?[^\S\n]*')
    display(Javascript(f"{_lb}({json.dumps(symbols)}, {_input}, {(_assets / 'layout.json').read_text()})"))
    display(HTML('<style>' + (_assets / 'fonts.css').read_text() +
                 '.bpl_choices button[aria-selected="true"]{background:#2463a8;color:white}</style>'))
    get_ipython().find_line_magic('bpl').__self__._loaded = True

    display(Javascript(r'''(()=>{
    const bar=document.querySelector('.ngn_lb'); if(bar) bar.hidden=false;
    let button=document.getElementById('bpl-symbol-toggle');
    if(!button){
        button=document.createElement('button'); button.id='bpl-symbol-toggle';
        button.type='button'; button.textContent='BPL';
        button.title='Show or hide BPL symbols';
        button.className='uk-btn uk-btn-sm uk-btn-default cursor-pointer';
        document.getElementById('settings').before(button);
    }
    button.setAttribute('aria-pressed',String(bar && !bar.hidden));
    button.onclick=()=>{
        const current=document.querySelector('.ngn_lb');
        if(current){current.hidden=!current.hidden;button.setAttribute('aria-pressed',String(!current.hidden));}
    };
    })();'''))



In [ ]:
def rebuild_bpl_browser():
    """Regenerate the native browser bootstrap after changing keyboard code or examples."""
    from unittest.mock import patch
    from basedpl import symbols
    from importlib.resources import files
    captured = []
    with patch('IPython.display.display', lambda obj: captured.append(obj)):
        _bpl_keyboard()
    scripts = [obj.data for obj in captured if isinstance(obj, Javascript)]
    styles = ''.join(obj.data.removeprefix('<style>').removesuffix('</style>') for obj in captured if isinstance(obj, HTML))
    layout = json.loads((files('basedpl') / 'layout.json').read_text())
    payload = ','.join(json.dumps(x, ensure_ascii=False) for x in (symbols, layout, bpl_examples))
    marker = '// BEGIN GENERATED BPL BOOTSTRAP'
    target = _bpl_craft_dir / 'CRAFT.js'
    base = target.read_text().split(marker)[0].rstrip()
    boot = '\n' + marker + '\n(() => {\n'
    boot += 'const start=()=>{\n'
    boot += 'if(!document.getElementById("bpl-craft-fonts")){const s=document.createElement("style");s.id="bpl-craft-fonts";s.textContent=' + json.dumps(styles) + ';document.head.append(s);}\n'
    boot += '\n;\n'.join(scripts) + '\n;\n'
    boot += '((symbols,layout,examples)=>window.bplCraftPalette(window.bplCraftCatalogue(symbols,layout),examples))(' + payload + ');\n'
    boot += '};if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",start,{once:true});else start();\n})();\n'
    target.write_text(base + '\n' + boot)


In [ ]:
def setup_bpl():
    """Refresh the BPL keyboard and palette without changing lesson variables."""
    from basedpl import symbols
    from importlib.resources import files
    _bpl_keyboard()
    script = (_bpl_craft_dir / 'CRAFT.js').read_text().split('// BEGIN GENERATED BPL BOOTSTRAP')[0]
    layout = json.loads((files('basedpl') / 'layout.json').read_text())
    size = dict(fontSize=19, buttonWidth=21, buttonHeight=26, gap=0.5)
    payload = ','.join(json.dumps(x, ensure_ascii=False) for x in (symbols, layout, bpl_examples, size))
    script += '\n;((symbols,layout,examples,size)=>{window.bplCraftPalette(window.bplCraftCatalogue(symbols,layout),examples,size);})(' + payload + ');'
    display(Javascript(script))
    get_ipython().find_line_magic('bpl').__self__._loaded = True

setup_bpl()
